<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/preprocessing_190326_edited.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this file, we
1. automate the data preprocessing and
2. create multiplex horizontal visibility graphs for a certain time window in a file (edge list + adjacency matrix)

Comments:
- For the code to work, it needs to be in the same folder as the .edf and .seizure files.
- Window size can be adjusted; in my case the computer didn't have enough RAM for more than 5s.

What is missing:
- the code does not process files without a corresponding .seizure file
- actual graph creation for the correct windows (one wictal and one interictal)

In [ ]:
!pip install ts2vg
!pip install mne
%cd /content
!git clone https://github.com/ionikim/epilepsy_pediatrics_EEG.git
%cd /content/epilepsy_pediatrics_EEG
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import json
from ts2vg import HorizontalVG
import scipy.sparse as sp
from scipy.sparse import save_npz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 55.9 MB/s eta 0:00:00
/content
Cloning into 'epilepsy_pediatrics_EEG'...
remote: Enumerating objects: 470, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 470 (delta 84), reused 79 (delta 51), pack-reused 326 (from 2)
Receiving objects: 100% (470/470), 45.23 MiB | 21.15 MiB/s, done.
Resolving deltas: 100% (191/191), done.
/content/epilepsy_pediatrics_EEG


In [ ]:
RAW_DIR = Path("/data/raw")

PROCESSED_DIR = Path("/data/processed")
LABELED_DIR   = PROCESSED_DIR / "labeled_signals"
FILEMETA_DIR  = PROCESSED_DIR / "file_metadata"

WINDOWS_DIR      = Path("/data/windows")
WINDOWSIG_DIR    = WINDOWS_DIR / "signals"
WINDOWMETA_DIR   = WINDOWS_DIR / "metadata"

for d in [LABELED_DIR, FILEMETA_DIR, WINDOWSIG_DIR, WINDOWMETA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
print("CWD:", Path.cwd())
print("RAW_DIR exists:", Path("data/raw").exists())
print("Files in raw:")
for p in Path("data/raw").glob("*"):
    print(p.name)

CWD: /content/epilepsy_pediatrics_EEG
RAW_DIR exists: True
Files in raw:
chb01_03.edf.seizures
chb01_03.edf
.gitkeep


In [ ]:
def save_json(data, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

In [ ]:
PROCESSED_DIR = Path("data/processed")

# Extract seizure start and length from a .seizures annotation file.
def get_seizure_period(annotation_file):
    with open(annotation_file, "rb") as f:
        byte_array = f.read()
    start = int(bin(byte_array[38])[2:] + bin(byte_array[41])[2:], 2)
    length = byte_array[49]
    return start, length


# Load an EDF file, create a DataFrame, and label seizure periods
def process_edf_with_labels(edf_file, seizure_file):
    raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)
    raw.filter(1., 45.)
    raw.set_eeg_reference('average')

    electrode_names = raw.ch_names
    sfreq = raw.info['sfreq']
    n_samples = len(raw.times)
    time_marks = np.arange(n_samples) / sfreq
    electrode_data = raw.get_data()

    df = pd.DataFrame(electrode_data.T, columns=electrode_names)
    df.index = time_marks
    df.index.name = 'Time (s)'

    seizure_start = None
    seizure_length = None
    seizure_end = None

    if seizure_file:
        seizure_start, seizure_length = get_seizure_period(seizure_file)
        seizure_end = seizure_start + seizure_length
        df["label"] = df.index.map(
            lambda t: "ictal" if seizure_start <= t <= seizure_end else "interictal"
        )
        has_seizure = True
    else:
        df["label"] = "interictal"
        has_seizure = False

    file_meta = {
        "edf_file": str(edf_file),
        "seizure_file": str(seizure_file) if seizure_file else None,
        "sampling_rate_hz": float(sfreq),
        "channel_names": list(electrode_names),
        "n_channels": int(len(electrode_names)),
        "n_samples": int(n_samples),
        "duration_s": float(n_samples / sfreq),
        "has_seizure": bool(has_seizure),
        "seizure_start_s": float(seizure_start) if seizure_start is not None else None,
        "seizure_length_s": float(seizure_length) if seizure_length is not None else None,
        "seizure_end_s": float(seizure_end) if seizure_end is not None else None,
    }

    return df, file_meta

In [ ]:
def extract_window(df, window_size_s=10, mode="ictal", strategy="first"):
    sfreq = int(round(1 / (df.index[1] - df.index[0])))
    window_size_samples = window_size_s * sfreq

    if mode == "ictal":
        indices = np.where(df["label"] == "ictal")[0]
    elif mode == "interictal":
        indices = np.where(df["label"] == "interictal")[0]
    else:
        raise ValueError("mode must be 'ictal' or 'interictal'")

    if len(indices) == 0:
        raise ValueError(f"No {mode} samples found.")

    valid_starts = [
        idx for idx in indices
        if idx + window_size_samples <= len(df)
        and (df["label"].iloc[idx : idx + window_size_samples] == mode).all()
    ]

    if not valid_starts:
        raise ValueError(f"No contiguous {mode} block of {window_size_s}s found.")

    if mode == "ictal":
        onset_idx = int(indices[0])
        offset_idx = int(indices[-1])

        if strategy == "first":
            # onset-based earliest valid ictal start
            start_idx = int(valid_starts[0])

        elif strategy == "middle":
            # seizure middle 기준으로 window 중심 맞추기
            center_idx = (onset_idx + offset_idx) // 2
            start_idx = center_idx - (window_size_samples // 2)

        elif strategy == "last":
            # seizure 끝부분 기준
            start_idx = offset_idx - window_size_samples + 1

        else:
            raise ValueError("For ictal, strategy must be 'first', 'middle', or 'last'")

        if start_idx < 0 or start_idx + window_size_samples > len(df):
            raise ValueError(f"Invalid ictal window for strategy='{strategy}'")

        if not (df["label"].iloc[start_idx : start_idx + window_size_samples] == "ictal").all():
            raise ValueError(f"Ictal window for strategy='{strategy}' is not fully ictal.")

    elif mode == "interictal":
        if strategy == "first":
            start_idx = int(valid_starts[0])
        elif strategy == "middle":
            start_idx = int(valid_starts[len(valid_starts) // 2])
        elif strategy == "last":
            start_idx = int(valid_starts[-1])
        else:
            raise ValueError("For interictal, strategy must be 'first', 'middle', or 'last'")

    window = df.iloc[start_idx : start_idx + window_size_samples]
    return window, start_idx

def process_all_files(
    pattern="*.seizures",
    window_size_s=10,
    ictal_strategy="middle",
    interictal_strategy="first"
):
    seizure_files = sorted(glob.glob(pattern))

    if not seizure_files:
        print(f"No files found matching: {pattern}")
        return {}, {}, pd.DataFrame(), pd.DataFrame()

    all_dataframes = {}
    windows = {}
    window_records = []
    log_records = []

    for i, seizure_file in enumerate(seizure_files):
        edf_file = seizure_file.replace(".seizures", "")

        try:
            # 1. full dataframe
            df, file_meta = process_edf_with_labels(edf_file, seizure_file)
            all_dataframes[edf_file] = df

            # full labeled df 저장
            file_stem = Path(edf_file).stem
            out_df = Path("data/processed/labeled_signals") / f"{file_stem}.parquet"
            out_df.parent.mkdir(parents=True, exist_ok=True)
            df.to_parquet(out_df)

            # 2. ictal / interictal window
            ictal_window, ictal_start_idx = extract_window(
                df,
                window_size_s=window_size_s,
                mode="ictal",
                strategy=ictal_strategy
            )

            interictal_window, interictal_start_idx = extract_window(
                df,
                window_size_s=window_size_s,
                mode="interictal",
                strategy=interictal_strategy
            )

            # 3. windows dict
            ictal_name = f"window_ictal_{i}"
            interictal_name = f"window_interictal_{i}"

            windows[ictal_name] = ictal_window
            windows[interictal_name] = interictal_window

            # window 저장
            win_dir = Path("data/windows/signals")
            win_dir.mkdir(parents=True, exist_ok=True)

            ictal_out = win_dir / f"{ictal_name}.parquet"
            interictal_out = win_dir / f"{interictal_name}.parquet"

            ictal_window.to_parquet(ictal_out)
            interictal_window.to_parquet(interictal_out)

            # 4. metadata rows
            sfreq = int(round(1 / (df.index[1] - df.index[0])))

            window_records.append({
                "window_id": ictal_name,
                "source_edf": edf_file,
                "label": "ictal",
                "selection_rule": ictal_strategy,
                "start_idx": ictal_start_idx,
                "end_idx": ictal_start_idx + len(ictal_window) - 1,
                "start_time_s": float(ictal_window.index[0]),
                "end_time_s": float(ictal_window.index[-1]),
                "n_samples": len(ictal_window),
                "window_size_s": window_size_s,
                "sampling_rate_hz": sfreq
            })

            window_records.append({
                "window_id": interictal_name,
                "source_edf": edf_file,
                "label": "interictal",
                "selection_rule": interictal_strategy,
                "start_idx": interictal_start_idx,
                "end_idx": interictal_start_idx + len(interictal_window) - 1,
                "start_time_s": float(interictal_window.index[0]),
                "end_time_s": float(interictal_window.index[-1]),
                "n_samples": len(interictal_window),
                "window_size_s": window_size_s,
                "sampling_rate_hz": sfreq
            })

            # 5. success log
            log_records.append({
                "edf_file": edf_file,
                "status": "success",
                "message": "ok",
                "saved_labeled_df": str(out_df),
                "saved_ictal_window": str(ictal_out),
                "saved_interictal_window": str(interictal_out)
            })

            # 6. summary
            print(f"\n[{i}] {edf_file}")
            print(f"  Has seizure                 : {file_meta['has_seizure']}")
            print(f"  Saved full df               : {out_df}")

            print(f"  Ictal strategy              : {ictal_strategy}")
            print(f"  Ictal window name           : {ictal_name}")
            print(f"  Ictal start idx             : {ictal_start_idx}")
            print(f"  Ictal end idx               : {ictal_start_idx + len(ictal_window) - 1}")
            print(f"  Ictal start time (s)        : {float(ictal_window.index[0])}")
            print(f"  Ictal end time (s)          : {float(ictal_window.index[-1])}")

            print(f"  Interictal strategy         : {interictal_strategy}")
            print(f"  Interictal window name      : {interictal_name}")
            print(f"  Interictal start idx        : {interictal_start_idx}")
            print(f"  Interictal end idx          : {interictal_start_idx + len(interictal_window) - 1}")
            print(f"  Interictal start time (s)   : {float(interictal_window.index[0])}")
            print(f"  Interictal end time (s)     : {float(interictal_window.index[-1])}")

        except FileNotFoundError as e:
            print(f"ERROR: EDF file not found for {seizure_file} — expected {edf_file}")
            log_records.append({
                "edf_file": edf_file,
                "status": "file_not_found",
                "message": str(e)
            })

        except ValueError as e:
            print(f"WARNING [{edf_file}]: {e}")
            log_records.append({
                "edf_file": edf_file,
                "status": "warning",
                "message": str(e)
            })

        except Exception as e:
            print(f"ERROR processing {seizure_file}: {e}")
            log_records.append({
                "edf_file": edf_file,
                "status": "error",
                "message": str(e)
            })

    windows_index = pd.DataFrame(window_records)
    out_index = Path("data/windows/metadata/windows_index.csv")
    out_index.parent.mkdir(parents=True, exist_ok=True)
    windows_index.to_csv(out_index, index=False)

    processing_log = pd.DataFrame(log_records)
    out_log = PROCESSED_DIR / "processing_log.csv"
    out_log.parent.mkdir(parents=True, exist_ok=True)
    processing_log.to_csv(out_log, index=False)

    print(f"\nSaved window index: {out_index}")
    print(f"Saved processing log: {out_log}")

    return all_dataframes, windows, windows_index, processing_log

In [ ]:
# Run it
all_dataframes, windows, windows_index, processing_log = process_all_files(
    pattern="data/raw/*.edf.seizures",
    window_size_s=5
)

# Access individual windows
# windows["window_ictal_0"]
# windows["window_interictal_2"]

# Check if all windows have the same number of samples
sizes = {name: len(df) for name, df in windows.items()}
print("\nWindow sizes:", sizes)

/tmp/ipykernel_8899/435486982.py:14: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 845 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.

[0] data/raw/chb01_03.edf
  Has seizure                 : True
  Saved full df               : data/processed/labeled_signals/chb01_03.parquet
  Ictal strategy              : middle
  Ictal window name           : window_ictal_0
  Ictal start idx             : 771456
  Ictal end idx               : 772735
  Ictal start time (

In [ ]:
display(windows["window_ictal_0"])
display(windows["window_interictal_0"])

,FP1-F7,F7-T7,T7-P7,P7-O1,FP1-F3,F3-C3,C3-P3,P3-O1,FP2-F4,F4-C4,...,T8-P8-0,P8-O2,FZ-CZ,CZ-PZ,P7-T7,T7-FT9,FT9-FT10,FT10-T8,T8-P8-1,label
Time (s),,,,,,,,,,,,,,,,,,,,,
3013.500000,0.000035,9.378768e-05,-0.000037,-2.026757e-05,-0.000007,0.000139,0.000001,-0.000062,-0.000049,0.000003,...,-0.000130,0.000030,0.000050,3.688826e-05,0.000021,-0.000065,-0.000002,0.000055,-0.000130,ictal
3013.503906,0.000047,9.948495e-05,-0.000041,-1.014135e-05,0.000027,0.000123,0.000002,-0.000058,-0.000096,0.000020,...,-0.000126,0.000041,0.000050,3.329896e-05,0.000035,-0.000058,0.000024,0.000030,-0.000126,ictal
3013.507812,0.000077,9.450175e-05,-0.000051,-9.812577e-06,0.000077,0.000098,-0.000004,-0.000061,-0.000076,0.000014,...,-0.000123,0.000047,0.000044,2.329079e-05,0.000043,-0.000056,0.000020,0.000012,-0.000123,ictal
3013.511719,0.000099,8.598956e-05,-0.000059,-1.489622e-05,0.000108,0.000077,-0.000012,-0.000062,-0.000014,-0.000014,...,-0.000133,0.000054,0.000037,1.085360e-05,0.000047,-0.000057,0.000002,0.000014,-0.000133,ictal
3013.515625,0.000104,8.267999e-05,-0.000060,-1.791437e-05,0.000113,0.000067,-0.000017,-0.000055,0.000035,-0.000043,...,-0.000140,0.000062,0.000035,3.727381e-07,0.000054,-0.000055,-0.000018,0.000027,-0.000140,ictal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3018.480469,0.000158,-4.971673e-07,-0.000091,-6.870288e-07,0.000243,-0.000147,-0.000076,0.000045,0.000127,0.000048,...,-0.000156,-0.000149,-0.000070,-1.220565e-04,0.000098,0.000022,0.000092,0.000084,-0.000156,ictal
3018.484375,0.000216,-1.948958e-05,-0.000088,-1.564833e-05,0.000306,-0.000165,-0.000078,0.000029,0.000131,0.000032,...,-0.000178,-0.000149,-0.000076,-1.236230e-04,0.000088,0.000037,0.000072,0.000090,-0.000178,ictal
3018.488281,0.000270,-2.023694e-05,-0.000101,-2.947990e-05,0.000376,-0.000165,-0.000102,0.000009,0.000140,0.000009,...,-0.000207,-0.000134,-0.000086,-1.259285e-04,0.000091,0.000029,0.000048,0.000098,-0.000207,ictal


,FP1-F7,F7-T7,T7-P7,P7-O1,FP1-F3,F3-C3,C3-P3,P3-O1,FP2-F4,F4-C4,...,T8-P8-0,P8-O2,FZ-CZ,CZ-PZ,P7-T7,T7-FT9,FT9-FT10,FT10-T8,T8-P8-1,label
Time (s),,,,,,,,,,,,,,,,,,,,,
0.000000,-7.218194e-21,1.252136e-21,-1.484149e-20,8.028399e-21,-1.399446e-20,1.819279e-20,-1.288963e-21,-1.060633e-20,-3.830062e-21,-1.568852e-20,...,-1.060633e-20,4.190972e-20,-3.830062e-21,1.311060e-20,6.334333e-21,-4.677095e-21,-3.830062e-21,-9.759293e-21,-1.060633e-20,interictal
0.003906,2.027821e-05,-1.952150e-05,1.026714e-05,2.114880e-06,8.436185e-06,4.736962e-06,-1.757520e-05,1.741806e-05,1.991319e-05,3.901279e-05,...,4.865140e-05,-1.142307e-04,2.072734e-05,-4.992945e-05,4.865270e-06,-5.012790e-06,1.349516e-05,3.428217e-05,4.865140e-05,interictal
0.007812,3.181107e-05,-3.056144e-05,1.605986e-05,3.111230e-06,1.307316e-05,7.570693e-06,-2.779998e-05,2.742782e-05,3.100897e-05,6.110447e-05,...,7.619164e-05,-1.782242e-04,3.236351e-05,-7.875416e-05,7.515170e-06,-8.528854e-06,2.122066e-05,5.387084e-05,7.619164e-05,interictal
0.011719,3.269548e-05,-3.077710e-05,1.676653e-05,3.086480e-06,1.338957e-05,8.205976e-06,-2.910764e-05,2.920421e-05,3.156265e-05,6.277518e-05,...,7.896025e-05,-1.848364e-04,3.315242e-05,-8.195742e-05,7.890227e-06,-1.069102e-05,2.274879e-05,5.634785e-05,7.896025e-05,interictal
0.015625,2.829422e-05,-2.515750e-05,1.534829e-05,3.018022e-06,1.203468e-05,7.744959e-06,-2.588431e-05,2.760407e-05,2.720113e-05,5.450926e-05,...,7.040122e-05,-1.672792e-04,2.882630e-05,-7.222593e-05,7.611872e-06,-1.224467e-05,2.203175e-05,5.102741e-05,7.040122e-05,interictal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4.980469,-7.920083e-06,-3.113402e-06,8.226804e-06,2.290847e-05,2.830897e-05,-6.097448e-06,8.374500e-06,-1.047864e-05,3.029140e-05,-1.940584e-05,...,-1.395058e-05,-2.207946e-05,7.291063e-06,7.937072e-06,-7.278360e-06,-3.657450e-05,6.116364e-05,1.958665e-05,-1.395058e-05,interictal
4.984375,-1.198343e-05,-3.709184e-06,1.125239e-05,1.624815e-05,1.976753e-05,-4.045718e-06,1.166494e-05,-1.627664e-05,1.120624e-05,-1.013205e-05,...,-1.036970e-05,-1.569797e-05,6.710007e-06,1.696751e-05,-1.153740e-05,-2.841548e-05,4.997037e-05,1.724661e-05,-1.036970e-05,interictal
4.988281,-1.393546e-05,-2.855600e-06,1.269850e-05,9.056515e-06,1.070004e-05,2.270860e-06,1.139595e-05,-2.022012e-05,-9.641808e-06,1.891190e-06,...,-5.160790e-06,-7.976283e-06,6.888711e-06,2.358507e-05,-1.486973e-05,-1.932127e-05,3.017660e-05,1.452264e-05,-5.160790e-06,interictal


In [ ]:
# build a multiplex horizontal visibility graph from an EEG window df
# -> each electrode is on one layer, each time point connects all electrodes (inter-layer edges)
# returns: edge list, adjacency matrix, node_index (dict for global node index), layer_info (dict with per-electrode HVG info)

def build_multiplex_hvg(window_df):
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes   = len(electrode_cols)
    n_timepoints   = len(window_df)
    n_nodes_total  = n_electrodes * n_timepoints

    print(f"Electrodes   : {n_electrodes}")
    print(f"Time points  : {n_timepoints}")
    print(f"Total nodes  : {n_nodes_total}")

    # 1. NODE INDEX: map (electrode_idx, time_idx) -> global node index
    # node layout: electrode 0 gets nodes 0..n_timepoints-1,
    #              electrode 1 gets nodes n_timepoints..2*n_timepoints-1, etc.

    def node_id(layer, t):
        return layer * n_timepoints + t

    node_index = {
        (electrode_cols[l], t): node_id(l, t)
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    }

    # 2. INTRA-LAYER EDGES: HVG edges within each electrode
    intra_edges = []
    layer_info  = {}

    for l, electrode in enumerate(electrode_cols):
        ts  = window_df[electrode].values
        hvg = HorizontalVG()
        hvg.build(ts)

        for (t_i, t_j) in hvg.edges:
          u_label = f"{electrode}_t{t_i}"
          v_label = f"{electrode}_t{t_j}"
          intra_edges.append({
            "Source": u_label,
            "Target": v_label,
            "source_label": u_label,
            "target_label": v_label,
            "edge_type": "intra",
            "layer": electrode,
        })

        layer_info[electrode] = {
            "n_intra_edges": len(hvg.edges),
            "layer_index"  : l,
        }

    print(f"Intra-layer edges : {len(intra_edges)}")

    # 3. INTER-LAYER EDGES: connect same time point across all electrodes
    # at each time point t, connect every electrode to every other (full coupling)

    inter_edges = []

    for t in range(n_timepoints):
      for l_i in range(n_electrodes):
          for l_j in range(l_i + 1, n_electrodes):
              u_label = f"{electrode_cols[l_i]}_t{t}"
              v_label = f"{electrode_cols[l_j]}_t{t}"
              inter_edges.append({
                  "Source": u_label,
                  "Target": v_label,
                  "source_label": u_label,
                  "target_label": v_label,
                  "edge_type": "inter",
                  "layer": f"{electrode_cols[l_i]}<->{electrode_cols[l_j]}",
             })

    print(f"Inter-layer edges : {len(inter_edges)}")

    # 4. COMBINED EDGE LIST
    edge_list = pd.DataFrame(intra_edges + inter_edges)
    print(f"Total edges       : {len(edge_list)}")

    # 5. ADJACENCY MATRIX
    # use sparse matrix for efficiency (n_nodes can be large)
    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.int8)

    for _, row in edge_list.iterrows():
        u, v = int(row["source"]), int(row["target"])
        adj_sparse[u, v] = 1
        adj_sparse[v, u] = 1  # undirected

    # convert to labeled DataFrame
    node_labels = [
        f"{electrode_cols[l]}_t{t}"
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    ]
    adj_matrix = pd.DataFrame(
        adj_sparse.toarray(),
        index   = node_labels,
        columns = node_labels
    )

    return edge_list, adj_matrix, node_index, layer_info

In [ ]:
# =========================================================
# 1. MAKE NODE TABLE
# =========================================================
def make_multiplex_node_table(window_df, graph_label):
    """
    Create node table for multiplex HVG.
    Each node = one electrode at one time point.
    """
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_timepoints = len(window_df)

    rows = []
    for electrode in electrode_cols:
        for t in range(n_timepoints):
            node_name = f"{electrode}_t{t}"
            rows.append({
                "Id": node_name,
                "Label": node_name,
                "electrode": electrode,
                "time_idx": t,
                "layer": electrode,
                "graph_label": graph_label
            })

    return pd.DataFrame(rows)


In [ ]:
# =========================================================
# 2. BUILD MULTIPLEX HVG
# =========================================================
def build_multiplex_hvg(window_df):
    """
    Build a multiplex horizontal visibility graph from an EEG window DataFrame.

    Each electrode = one layer
    Each time point across electrodes = fully connected inter-layer clique

    Returns:
    - edge_list    : DataFrame with Source / Target / edge_type / layer
    - adj_sparse   : scipy sparse adjacency matrix
    - node_index   : dict mapping (electrode, time_idx) -> node label
    - layer_info   : dict with per-electrode HVG info
    - node_labels  : list of node labels in adjacency order
    """
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes = len(electrode_cols)
    n_timepoints = len(window_df)
    n_nodes_total = n_electrodes * n_timepoints

    print(f"Electrodes        : {n_electrodes}")
    print(f"Time points       : {n_timepoints}")
    print(f"Total nodes       : {n_nodes_total}")

    def node_id(electrode, t):
        return f"{electrode}_t{t}"

    node_labels = [
        node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    ]

    label_to_idx = {label: idx for idx, label in enumerate(node_labels)}

    node_index = {
        (electrode, t): node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    }

    # -----------------------------------------------------
    # 1) intra-layer edges
    # -----------------------------------------------------
    intra_rows = []
    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.int8)
    layer_info = {}

    for l, electrode in enumerate(electrode_cols):
        ts = window_df[electrode].values
        hvg = HorizontalVG()
        hvg.build(ts)

        for (t_i, t_j) in hvg.edges:
            u = node_id(electrode, t_i)
            v = node_id(electrode, t_j)

            intra_rows.append({
                "Source": u,
                "Target": v,
                "source_label": u,
                "target_label": v,
                "edge_type": "intra",
                "layer": electrode
            })

            i = label_to_idx[u]
            j = label_to_idx[v]
            adj_sparse[i, j] = 1
            adj_sparse[j, i] = 1

        layer_info[electrode] = {
            "n_intra_edges": len(hvg.edges),
            "layer_index": l
        }

    print(f"Intra-layer edges : {len(intra_rows)}")

    # -----------------------------------------------------
    # 2) inter-layer edges
    # -----------------------------------------------------
    inter_rows = []

    for t in range(n_timepoints):
        for i in range(n_electrodes):
            for j in range(i + 1, n_electrodes):
                e_i = electrode_cols[i]
                e_j = electrode_cols[j]

                u = node_id(e_i, t)
                v = node_id(e_j, t)

                inter_rows.append({
                    "Source": u,
                    "Target": v,
                    "source_label": u,
                    "target_label": v,
                    "edge_type": "inter",
                    "layer": f"{e_i}<->{e_j}"
                })

                ui = label_to_idx[u]
                vj = label_to_idx[v]
                adj_sparse[ui, vj] = 1
                adj_sparse[vj, ui] = 1

    print(f"Inter-layer edges : {len(inter_rows)}")

    # -----------------------------------------------------
    # 3) combined edge list
    # -----------------------------------------------------
    edge_list = pd.DataFrame(intra_rows + inter_rows)
    print(f"Total edges       : {len(edge_list)}")

    return edge_list, adj_sparse.tocsr(), node_index, layer_info, node_labels

In [ ]:
# =========================================================
# 3. SAVE OUTPUTS
# =========================================================
def save_multiplex_hvg_outputs(
    edge_list,
    adj_sparse,
    node_index,
    node_labels,
    layer_info,
    window_df,
    graph_id,
    source_edf,
    window_id,
    label,
    out_root="data/graphs"
):
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    edges_dir = out_root / "edges"
    nodes_dir = out_root / "nodes"
    adj_dir = out_root / "adjacency_sparse"
    meta_dir = out_root / "metadata"

    for d in [edges_dir, nodes_dir, adj_dir, meta_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # 1) node table
    node_df = make_multiplex_node_table(window_df, graph_label=label)

    # 2) save edge list
    edge_path = edges_dir / f"{graph_id}_edges.csv"
    edge_list.to_csv(edge_path, index=False)

    # 3) save node list
    node_path = nodes_dir / f"{graph_id}_nodes.csv"
    node_df.to_csv(node_path, index=False)

    # 4) save sparse adjacency
    adj_path = adj_dir / f"{graph_id}_adjacency_sparse.npz"
    save_npz(adj_path, adj_sparse)

    # 5) save node labels for adjacency reconstruction
    node_labels_path = meta_dir / f"{graph_id}_node_labels.json"
    with open(node_labels_path, "w", encoding="utf-8") as f:
        json.dump(node_labels, f, indent=2)

    # 6) save layer info
    layer_info_df = (
        pd.DataFrame(layer_info)
        .T
        .reset_index()
        .rename(columns={"index": "electrode"})
    )
    layer_info_path = meta_dir / f"{graph_id}_layer_info.csv"
    layer_info_df.to_csv(layer_info_path, index=False)

    # 7) save node index
    node_index_str = {f"{k[0]}__t{k[1]}": v for k, v in node_index.items()}
    node_index_path = meta_dir / f"{graph_id}_node_index.json"
    with open(node_index_path, "w", encoding="utf-8") as f:
        json.dump(node_index_str, f, indent=2)

    # 8) save metadata
    metadata = {
        "graph_id": graph_id,
        "source_edf": source_edf,
        "window_id": window_id,
        "label": label,
        "n_nodes": int(node_df.shape[0]),
        "n_edges": int(edge_list.shape[0]),
        "n_intra_edges": int((edge_list["edge_type"] == "intra").sum()),
        "n_inter_edges": int((edge_list["edge_type"] == "inter").sum()),
        "n_layers": int(node_df["electrode"].nunique()),
        "n_timepoints": int(len(window_df)),
        "adjacency_format": "scipy_sparse_npz"
    }

    metadata_path = meta_dir / f"{graph_id}_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return {
        "edges": str(edge_path),
        "nodes": str(node_path),
        "adjacency_sparse": str(adj_path),
        "node_labels": str(node_labels_path),
        "layer_info": str(layer_info_path),
        "node_index": str(node_index_path),
        "metadata": str(metadata_path),
    }

In [ ]:
# =========================================================
# 4. RUN ICTAL / INTERICTAL
# =========================================================

# -------------------------
# ICTAL
# -------------------------
ictal_key = next(k for k in windows if k.startswith("window_ictal_"))
ictal_window = windows[ictal_key]

ictal_edge_list, ictal_adj_sparse, ictal_node_index, ictal_layer_info, ictal_node_labels = build_multiplex_hvg(
    ictal_window
)

ictal_graph_id = f"chb01_03_{ictal_key}"

saved_paths_ictal = save_multiplex_hvg_outputs(
    edge_list=ictal_edge_list,
    adj_sparse=ictal_adj_sparse,
    node_index=ictal_node_index,
    node_labels=ictal_node_labels,
    layer_info=ictal_layer_info,
    window_df=ictal_window,
    graph_id=ictal_graph_id,
    source_edf="data/raw/chb01_03.edf",
    window_id=ictal_key,
    label="ictal",
    out_root="data/graphs"
)

# -------------------------
# INTERICTAL
# -------------------------
interictal_key = next(k for k in windows if k.startswith("window_interictal_"))
interictal_window = windows[interictal_key]

interictal_edge_list, interictal_adj_sparse, interictal_node_index, interictal_layer_info, interictal_node_labels = build_multiplex_hvg(
    interictal_window
)

interictal_graph_id = f"chb01_03_{interictal_key}"

saved_paths_interictal = save_multiplex_hvg_outputs(
    edge_list=interictal_edge_list,
    adj_sparse=interictal_adj_sparse,
    node_index=interictal_node_index,
    node_labels=interictal_node_labels,
    layer_info=interictal_layer_info,
    window_df=interictal_window,
    graph_id=interictal_graph_id,
    source_edf="data/raw/chb01_03.edf",
    window_id=interictal_key,
    label="interictal",
    out_root="data/graphs"
)

print("ICTAL FILES:")
print(saved_paths_ictal)

print("\nINTERICTAL FILES:")
print(saved_paths_interictal)

Electrodes        : 23
Time points       : 1280
Total nodes       : 29440
Intra-layer edges : 57974
Inter-layer edges : 323840
Total edges       : 381814
Electrodes        : 23
Time points       : 1280
Total nodes       : 29440
Intra-layer edges : 57827
Inter-layer edges : 323840
Total edges       : 381667
ICTAL FILES:
{'edges': 'data/graphs/edges/chb01_03_window_ictal_0_edges.csv', 'nodes': 'data/graphs/nodes/chb01_03_window_ictal_0_nodes.csv', 'adjacency_sparse': 'data/graphs/adjacency_sparse/chb01_03_window_ictal_0_adjacency_sparse.npz', 'node_labels': 'data/graphs/metadata/chb01_03_window_ictal_0_node_labels.json', 'layer_info': 'data/graphs/metadata/chb01_03_window_ictal_0_layer_info.csv', 'node_index': 'data/graphs/metadata/chb01_03_window_ictal_0_node_index.json', 'metadata': 'data/graphs/metadata/chb01_03_window_ictal_0_metadata.json'}

INTERICTAL FILES:
{'edges': 'data/graphs/edges/chb01_03_window_interictal_0_edges.csv', 'nodes': 'data/graphs/nodes/chb01_03_window_interictal_

In [ ]:
import shutil
from pathlib import Path

src_dir = Path("data/graphs")
zip_base = "graphs_export"

shutil.make_archive(zip_base, "zip", root_dir=src_dir)
print("Created:", f"{zip_base}.zip")

Created: graphs_export.zip


In [ ]:
from google.colab import files
files.download("graphs_export.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>